# Hugging Face 모델 탐색 실습

이번 실습에서는 Hugging Face에서 제공하는 다양한 인공지능 모델을 직접 찾아보고, Colab에서 실행해보겠습니다.

오늘은 모델을 직접 학습시키는 것이 아니라, 이미 학습되어 공개된 모델을 불러와 사용하는 방법을 연습합니다.

이번 시간에 실습할 task는 다음 4가지입니다.

1. 감정분석
2. 번역
3. 한국어 요약
4. 문장 유사도 비교


#라이브러리 설치

Hugging Face 모델을 사용하기 위해 필요한 라이브러리를 먼저 설치하겠습니다.

아래 코드는 실습에 필요한 기본 라이브러리를 설치하는 코드입니다.
처음 한 번만 실행하면 됩니다.

transformers: Hugging Face 모델을 불러오고 실행하는 라이브러리입니다.
sentencepiece: 일부 번역 및 요약 모델에서 사용하는 토크나이저 라이브러리입니다.
sentence-transformers: 문장을 벡터로 바꾸고 유사도를 계산할 때 사용하는 라이브러리입니다.

In [ ]:
!pip install -q transformers sentencepiece sentence-transformers accelerate

In [ ]:
import transformers
import sentence_transformers

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

transformers: 5.12.1
sentence-transformers: 5.6.0


#Hugging Face 모델 검색 방법

Hugging Face에는 다양한 인공지능 모델이 공개되어 있습니다.

이번 실습에서는 원하는 task에 맞는 모델을 직접 검색해보고, Colab에서 실행해보겠습니다.

모델을 찾는 순서는 다음과 같습니다.

1. Hugging Face 사이트에 접속합니다.
2. 상단 메뉴에서 `Models`를 클릭합니다.
3. 왼쪽 메뉴에서 원하는 `Task`를 선택합니다.
4. 검색창에 원하는 키워드를 입력합니다.
5. 마음에 드는 모델 페이지에 들어갑니다.
6. 모델 이름을 복사합니다.
7. Colab 코드의 `model_name` 부분에 붙여넣습니다.

모델 이름은 보통 다음과 같은 형태입니다.

```text
사용자명/모델명
```

예를 들어 다음과 같은 이름을 사용할 수 있습니다.

```text
distilbert-base-uncased-finetuned-sst-2-english
eenzeenee/t5-base-korean-summarization
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

처음에는 어떤 모델을 골라야 할지 헷갈릴 수 있습니다.
그럴 때는 다운로드 수가 많거나, 모델 카드에 사용 예시가 잘 적혀 있는 모델을 먼저 선택해보면 좋습니다.


#1. 감정분석
- sentiment-analysis
- text-classificatio

In [ ]:
from transformers import pipeline

pipe = pipeline("text-classification", model="nlp04/korean_sentiment_analysis_kcelectra")

config.json:   0%|          | 0.00/1.87k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/511M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/511M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/450k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
output=pipe("다시는 이용하고 싶지 않습니다.")
print(output)

[{'label': 'sadness', 'score': 0.6333580613136292}]


In [ ]:
pipe.model.config.id2label

{0: '기쁨(행복한)',
 1: '고마운',
 2: '설레는(기대하는)',
 3: '사랑하는',
 4: '즐거운(신나는)',
 5: '일상적인',
 6: '생각이 많은',
 7: '슬픔(우울한)',
 8: '힘듦(지침)',
 9: '짜증남',
 10: '걱정스러운(불안한)'}

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("tabularisai/multilingual-emotion-classification")
model = AutoModelForSequenceClassification.from_pretrained("tabularisai/multilingual-emotion-classification")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

#2. 번역
- translation

In [ ]:
from transformers import MT5ForConditionalGeneration, AutoTokenizer

model_name = "EphAsad/Midas-Korean-English"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = MT5ForConditionalGeneration.from_pretrained(model_name)

def translate(text, direction="ko-en"):
    if direction == "ko-en":
        prefix = "translate Korean to English: "
    else:
        prefix = "translate English to Korean: "

    inputs = tokenizer(
        prefix + text,
        return_tensors = "pt",
        max_length     = 512,
        truncation     = True,
    )
    outputs = model.generate(
        **inputs,
        max_new_tokens     = 512,
        num_beams          = 4,
        early_stopping     = True,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Korean → English
print(translate("인공지능은 세상을 변화시키고 있습니다.", direction="ko-en"))

# English → Korean
print(translate("Artificial intelligence is changing th world.", direction="en-ko"))


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial Intelligence changes the world.
과학적 정보는 전 세계에 변화하는 것입니다.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ko-en")
model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-ko-en")

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [ ]:
# Korean → English
print(translate("인공지능은 세상을 변화시키고 있습니다.", direction="ko-en"))

# English → Korean
print(translate("Artificial intelligence is changing th world.", direction="en-ko"))

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial Intelligence changes the world.
과학적 정보는 전 세계에 변화하는 것입니다.


#3. 요약
- summarization
- text2text-generation

In [ ]:
!pip uninstall -y transformers tokenizers
!pip install "transformers==4.44.2" "tokenizers==0.19.1" sentencepiece accelerate

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 655.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.20.1
    Uninstalling huggingface_hub-1.20.1:
      Successfully uninstalled huggingface_hub-1.20.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you hav

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model="eenzeenee/t5-base-korean-summarization")

config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    task="summarization",
    model="ainize/kobart-news"
)

text = """
오늘 학교에서는 인공지능 특강이 열렸다.
학생들은 머신러닝과 딥러닝의 차이, 자연어 처리 기술,
생성형 AI의 활용 사례를 배우고 직접 실습을 진행하였다.
"""

result = summarizer(text, max_length=40, min_length=15, do_sample=False)

print(result)

KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    task="summarization",
    model="ainize/kobart-news"
)

text = """
인공지능 기술은 의료, 교육, 금융 등 다양한 분야에서 활용되고 있다.
특히 생성형 AI는 문서 작성, 번역, 이미지 생성 등 다양한 작업을 수행할 수 있다.
앞으로 인공지능 기술은 더욱 발전하여 우리의 삶을 크게 변화시킬 것으로 기대된다.
"""

result = summarizer(text, max_length=50, min_length=20, do_sample=False)

print(result)

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

#4. 문장 유사도 비교
- sentence-similarity
- sentence-transformers 라이브러리 사용

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("dragonkue/multilingual-e5-small-ko")

sentences = [
    'query: 북한가족법 몇 차 개정에서 이혼판결 확정 후 3개월 내에 등록시에만 유효하다는 조항을 확실히 했을까?',
    'passage: 1990년에 제정된 북한 가족법은 지금까지 4차례 개정되어 현재에 이르고 있다. 1993년에 이루어진 제1차 개정은 주로 규정의 정확성을 기하기 위하여 몇몇 조문을 수정한 것이며, 실체적인 내용을 보완한 것은 상속의 승인과 포기기간을 설정한 제52조 정도라고 할 수 있다. 2004년에 이루어진 제2차에 개정에서는 제20조제3항을 신설하여 재판상 확정된 이혼판결을 3개월 내에 등록해야 이혼의 효력이 발생한다는 것을 명확하게 하였다. 2007년에 이루어진 제3차 개정에서는 부모와 자녀 관계 또한 신분등록기관에 등록한 때부터 법적 효력이 발생한다는 것을 신설(제25조제2항)하였다. 또한 미성년자, 노동능력 없는 자의 부양과 관련(제37조제2항)하여 기존에는 “부양능력이 있는 가정성원이 없을 경우에는 따로 사는 부모나 자녀, 조부모나 손자녀, 형제자매가 부양한다”고 규정하고 있었던 것을 “부양능력이 있는 가정성원이 없을 경우에는 따로 사는 부모나 자녀가 부양하며 그들이 없을 경우에는 조부모나 손자녀, 형제자매가 부양한다”로 개정하였다.',
    'passage: 환경마크 제도, 인증기준 변경으로 기업부담 줄인다\n환경마크 제도 소개\n□ 개요\n○ 동일 용도의 다른 제품에 비해 ‘제품의 환경성*’을 개선한 제품에 로고와 설명을 표시할 수 있도록하는 인증 제도\n※ 제품의 환경성 : 재료와 제품을 제조․소비 폐기하는 전과정에서 오염물질이나 온실가스 등을 배출하는 정도 및 자원과 에너지를 소비하는 정도 등 환경에 미치는 영향력의 정도(「환경기술 및 환경산업 지원법」제2조제5호)\n□ 법적근거\n○ 「환경기술 및 환경산업 지원법」제17조(환경표지의 인증)\n□ 관련 국제표준\n○ ISO 14024(제1유형 환경라벨링)\n□ 적용대상\n○ 사무기기, 가전제품, 생활용품, 건축자재 등 156개 대상제품군\n□ 인증현황\n○ 2,737개 기업의 16,647개 제품(2015.12월말 기준)',
]
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# Get the similarity scores for the embeddings
similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

ImportError: cannot import name 'find_pruneable_heads_and_indices' from 'transformers.pytorch_utils' (/usr/local/lib/python3.12/dist-packages/transformers/pytorch_utils.py)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium."
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

ImportError: cannot import name 'find_pruneable_heads_and_indices' from 'transformers.pytorch_utils' (/usr/local/lib/python3.12/dist-packages/transformers/pytorch_utils.py)